# Data Cleaning & Preparation
## Enterprise Predictive Analytics Engine — E-commerce Domain

**Objective:** Load, merge, and clean the Olist Brazilian E-commerce dataset
into a single analysis-ready master table for downstream exploratory
analysis, clustering, classification, and forecasting.

**Input:** 9 raw CSV files in `data/raw/`
**Output:** `data/processed/olist_master_cleaned.csv`

**Steps:**
1. Load raw data
2. Convert date columns
3. Merge into master table
4. Engineer key features
5. Handle missing values
6. Save cleaned output

## 1. Import Libraries & Load Raw Data

Loading all 7 relevant tables from the Olist dataset. We inspect shapes
immediately to confirm the data loaded correctly before doing any
transformation.

In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "../data/raw/"

orders = pd.read_csv(DATA_PATH + "olist_orders_dataset.csv")
customers = pd.read_csv(DATA_PATH + "olist_customers_dataset.csv")
order_items = pd.read_csv(DATA_PATH + "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_PATH + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_PATH + "olist_order_reviews_dataset.csv")
products = pd.read_csv(DATA_PATH + "olist_products_dataset.csv")
category_translation = pd.read_csv(DATA_PATH + "product_category_name_translation.csv")

print("orders:", orders.shape)
print("customers:", customers.shape)
print("order_items:", order_items.shape)
print("payments:", payments.shape)
print("reviews:", reviews.shape)
print("products:", products.shape)

orders: (99441, 8)
customers: (99441, 5)
order_items: (112650, 7)
payments: (103886, 5)
reviews: (99224, 7)
products: (32951, 9)


## 2. Convert Date Columns

Order timestamps are loaded as strings by default. Converting them to
`datetime` is required before we can calculate delivery times, delays,
or aggregate by month for later analysis.

In [2]:
date_cols = ["order_purchase_timestamp", "order_approved_at",
             "order_delivered_carrier_date", "order_delivered_customer_date",
             "order_estimated_delivery_date"]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

orders[date_cols].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

## 3. Merge Into a Single Master Table

Joining orders → customers → order items → products → payments → reviews
on their respective keys. This creates one row per order item, enriched
with customer, product, payment, and review information — the foundation
for all later analysis.

In [5]:
df = orders.merge(customers, on="customer_id", how="left")
df = df.merge(order_items, on="order_id", how="left")

products = products.merge(category_translation, on="product_category_name", how="left")
df = df.merge(products[["product_id", "product_category_name_english"]], on="product_id", how="left")

payments_agg = payments.groupby("order_id").agg(
    payment_value=("payment_value", "sum"),
    payment_installments=("payment_installments", "max"),
    payment_type=("payment_type", "first")
).reset_index()
df = df.merge(payments_agg, on="order_id", how="left")

reviews_agg = reviews.groupby("order_id").agg(review_score=("review_score", "mean")).reset_index()
df = df.merge(reviews_agg, on="order_id", how="left")

print("Merged shape:", df.shape)
df.head()

Merged shape: (113425, 23)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name_english,payment_value,payment_installments,payment_type,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,housewares,38.71,1.0,credit_card,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumery,141.46,1.0,boleto,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,auto,179.12,3.0,credit_card,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,pet_shop,72.20,1.0,credit_card,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,stationery,28.62,1.0,credit_card,5.0


## 4. Feature Engineering

Creating derived fields that will be reused across modules:
- `delivery_delay_days` — actual vs estimated delivery (negative = early)
- `delivery_time_days` — total delivery duration
- `order_month` — used for monthly aggregation in EDA and forecasting

In [6]:
df["delivery_delay_days"] = (df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]).dt.days
df["delivery_time_days"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.days
df["order_month"] = df["order_purchase_timestamp"].dt.to_period("M").astype(str)

df[["delivery_delay_days", "delivery_time_days", "order_month"]].head()

,delivery_delay_days,delivery_time_days,order_month
0,-8.0,8.0,2017-10
1,-6.0,13.0,2018-07
2,-18.0,9.0,2018-08
3,-13.0,13.0,2017-11
4,-10.0,2.0,2018-02


## 5. Handle Missing Values

Checking missing data before deciding how to treat it — avoiding a
blanket `dropna()` that would silently discard usable rows. Only rows
missing a core field (purchase timestamp) are dropped; other gaps are
imputed or flagged.

In [7]:
print("Missing values before cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])

df = df.dropna(subset=["order_purchase_timestamp"])
df["review_score"] = df["review_score"].fillna(df["review_score"].median())
df["product_category_name_english"] = df["product_category_name_english"].fillna("unknown")
df["is_delivered"] = df["order_status"] == "delivered"

print("\nFinal cleaned shape:", df.shape)

Missing values before cleaning:
order_approved_at                 161
order_delivered_carrier_date     1968
order_delivered_customer_date    3229
order_item_id                     775
product_id                        775
seller_id                         775
shipping_limit_date               775
price                             775
freight_value                     775
product_category_name_english    2402
payment_value                       3
payment_installments                3
payment_type                        3
review_score                      961
delivery_delay_days              3229
delivery_time_days               3229
dtype: int64

Final cleaned shape: (113425, 27)


## 6. Save Cleaned Dataset

Exporting the cleaned master table to `data/processed/` so it can be
loaded directly by the EDA, clustering, classification, and forecasting
notebooks without repeating this pipeline.

In [8]:
df.to_csv("../data/processed/olist_master_cleaned.csv", index=False)
print("Saved to data/processed/olist_master_cleaned.csv")
df.columns.tolist()

Saved to data/processed/olist_master_cleaned.csv


['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value',
 'product_category_name_english',
 'payment_value',
 'payment_installments',
 'payment_type',
 'review_score',
 'delivery_delay_days',
 'delivery_time_days',
 'order_month',
 'is_delivered']

## Summary

- Merged 6 raw tables into one master dataset
- Engineered delivery and time-based features
- Handled missing values in review scores and product categories
- Output saved to `data/processed/olist_master_cleaned.csv`

**Next notebook:** `02_exploratory_data_analysis.ipynb` — exploring trends,
categories, geography, and delivery/review relationships before any
modeling begins.